In [17]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split

In [18]:
spark = (
     SparkSession.builder
    .appName("config-streaming")
    .master("spark://spark-master:7077")
    .config("spark.executor.memory", "2g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

In [19]:
lines = (
     spark.readStream
    .format("socket")
    .option("host", "localhost")
    .option("port", 9999)
    .load()
)

In [20]:
words = lines.select(
    explode(
        split(lines.value, " ")
    ).alias("word")
)

In [21]:
wordCounts = words.groupBy("word").count()

In [26]:
query = (
     wordCounts.writeStream
    .outputMode("complete")
    .format("console")
    .start()
)

-------------------------------------------
Batch: 0
-------------------------------------------
+----+-----+
|word|count|
+----+-----+
+----+-----+



-------------------------------------------
Batch: 1
-------------------------------------------
+---------+-----+
|     word|count|
+---------+-----+
|zieloność|    1|
|   Patrzę|    1|
|        —|    3|
| kurhanu;|    1|
|   śliską|    1|
|     kędy|    1|
|      Już|    1|
|    drogi|    1|
|   weszła|    1|
| powodzi,|    1|
|       by|    1|
|    ciszy|    1|
|  zapada,|    1|
|  kwiatów|    1|
| koralowe|    1|
|  natężam|    1|
|Wpłynąłem|    1|
|  nigdzie|    1|
|  brodzi,|    1|
|     ucho|    1|
+---------+-----+
only showing top 20 rows



In [27]:
query.stop()

In [28]:
spark.stop()